# Random Forest & Boosted Tree Models
**Jennifer Eigo - University of Connecticut - Dept. of Operations and Information Management**

-------------------------------------
Although basic tree models are great, they become even more powerful if we combine trees to improve performance.  Two common ways to do this is with a boosted tree and a random forest.  


# Environment Setup

In [ ]:
# import modules

import pandas as pd # for data viz and wrangling
import numpy as np # for 'numeric python'
import matplotlib.pyplot as plt # for data viz (more complex than pylab)
import seaborn as sns
from pylab import * # for data viz (import * means 'import all of the functions')

from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn import metrics
from scipy import stats
import statsmodels.api as sm


In [ ]:
# mount your google drive
from google.colab import drive
drive.mount('/content/drive')

#Load the Data - ToyotaCorolla1000

In [ ]:
# read data
# on the lefthand side, navigate to your data and copy the path
df = pd.read_csv('/content/drive/MyDrive/OPIM 5604 Python/Module 8/ToyotaCorolla1000.csv')

In [ ]:
# shape
# shows how many rows and columns
# this sample has 1000 rows and 10 columns
df.shape

In [ ]:
# Preview
print(df.head())

In [ ]:
# Create dummy variables for 'Fuel Type'
df = pd.get_dummies(df, columns=['Fuel Type'], drop_first=True)

# Display the first few rows to see the changes
print(df.head())

Before we model, we need to partition the data.

In [ ]:
from sklearn.model_selection import train_test_split

# Define the target variable
target = 'Price'
features = df.drop(target, axis=1)
target_variable = df[target]

# Split the data into 80% training and 20% test
X_train, X_test, y_train, y_test = train_test_split(features, target_variable, test_size=0.2, random_state=42)

# Split the 80% training data into 50% training and 30% validation
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.375, random_state=42) # 0.375 * 0.8 = 0.3

# Print the shapes of the resulting datasets
print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Test set shape:", X_test.shape, y_test.shape)

#Regression Tree

Let's start with just a simple regression tree to serve as a comparison point for our more complicated tree models. We are going straight to hyperparameter tuning because we already know that the maximal tree will be overfit to the training data.  

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, make_scorer

# Define the parameter grid
param_grid_reg = {
    'max_depth': range(2, 21),
    'min_samples_split': range(2, 21),
    'min_samples_leaf': range(1, 11)
}

# Create a scorer for negative mean squared error (GridSearchCV maximizes scores)
neg_mse_scorer = make_scorer(mean_squared_error, greater_is_better=False)

# Instantiate the Decision Tree Regressor model
reg_tree_model = DecisionTreeRegressor(random_state=42)

# Instantiate GridSearchCV
# Use the original reg_tree_model as the estimator
grid_search_reg = GridSearchCV(reg_tree_model, param_grid_reg, cv=5, scoring=neg_mse_scorer, n_jobs=-1)

# Fit GridSearchCV to the training data
grid_search_reg.fit(X_train, y_train)

# Print the best hyperparameters and best cross-validation score
print("Best hyperparameters found:", grid_search_reg.best_params_)
# Convert the best score back to positive MSE or RMSE for easier interpretation
best_mse = -grid_search_reg.best_score_
best_rmse = np.sqrt(best_mse)
print(f"Best cross-validation RMSE: {best_rmse:.4f}")

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Retrieve the best hyperparameters for the regression tree
best_params_reg = grid_search_reg.best_params_
print("Best hyperparameters for Regression Tree:", best_params_reg)

# Instantiate a new Decision Tree Regressor with the best hyperparameters
tuned_reg_tree_model = DecisionTreeRegressor(**best_params_reg, random_state=42)

# Train the tuned model on the complete training dataset
tuned_reg_tree_model.fit(X_train, y_train)

# Calculate the number of splits and leaves for the tuned tree
tree_tuned_reg = tuned_reg_tree_model.tree_
total_nodes_tuned_reg = tree_tuned_reg.node_count
leaf_nodes_tuned_reg = tree_tuned_reg.n_leaves
number_of_splits_tuned_reg = total_nodes_tuned_reg - leaf_nodes_tuned_reg

print(f"\nTuned Regression Tree:")
print(f"  Number of splits (internal nodes): {number_of_splits_tuned_reg}")
print(f"  Number of leaves: {leaf_nodes_tuned_reg}")


# Make predictions on training set
y_train_pred_tuned_reg = tuned_reg_tree_model.predict(X_train)

# Make predictions on validation set
y_val_pred_tuned_reg = tuned_reg_tree_model.predict(X_val)

# Make predictions on test set
y_test_pred_tuned_reg = tuned_reg_tree_model.predict(X_test)

# Calculate and print metrics for Training Set
rmse_train_tuned_reg = np.sqrt(mean_squared_error(y_train, y_train_pred_tuned_reg))
r2_train_tuned_reg = r2_score(y_train, y_train_pred_tuned_reg)

print("\nTuned Regression Tree Performance on Training Set:")
print(f"  RMSE: {rmse_train_tuned_reg:.4f}")
print(f"  R-squared: {r2_train_tuned_reg:.4f}")

# Calculate and print metrics for Validation Set
rmse_val_tuned_reg = np.sqrt(mean_squared_error(y_val, y_val_pred_tuned_reg))
r2_val_tuned_reg = r2_score(y_val, y_val_pred_tuned_reg)

print("\nTuned Regression Tree Performance on Validation Set:")
print(f"  RMSE: {rmse_val_tuned_reg:.4f}")
print(f"  R-squared: {r2_val_tuned_reg:.4f}")


# Calculate and print metrics for Test Set
rmse_test_tuned_reg = np.sqrt(mean_squared_error(y_test, y_test_pred_tuned_reg))
r2_test_tuned_reg = r2_score(y_test, y_test_pred_tuned_reg)

print("\nTuned Regression Tree Performance on Test Set:")
print(f"  RMSE: {rmse_test_tuned_reg:.4f}")
print(f"  R-squared: {r2_test_tuned_reg:.4f}")

Now we have the performace.  Let's look at the feature importance.

In [ ]:
import pandas as pd

# Get the feature importances from the tuned regression tree model
feature_importances = tuned_reg_tree_model.feature_importances_

# Get the feature names from the training data
feature_names = X_train.columns

# Create a pandas Series to easily view feature importances with their names
feature_importance_series = pd.Series(feature_importances, index=feature_names)

# Sort the feature importances in descending order
sorted_feature_importances = feature_importance_series.sort_values(ascending=False)

# Print the sorted feature importances
print("Feature Importances for Tuned Regression Tree:")
print(sorted_feature_importances)

#Random Forest

Now let's build a random forest.  We will start with a basic one. By default it builds the forest with 100 trees.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Instantiate a Random Forest Regressor model
rf_model = RandomForestRegressor(random_state=42)

# Train the model on the training data
rf_model.fit(X_train, y_train)

# Make predictions on training set
y_train_pred_rf = rf_model.predict(X_train)

# Make predictions on validation set
y_val_pred_rf = rf_model.predict(X_val)

# Make predictions on test set
y_test_pred_rf = rf_model.predict(X_test)

# Calculate and print metrics for Training Set
rmse_train_rf = np.sqrt(mean_squared_error(y_train, y_train_pred_rf))
r2_train_rf = r2_score(y_train, y_train_pred_rf)

print("Random Forest Performance on Training Set:")
print(f"  RMSE: {rmse_train_rf:.4f}")
print(f"  R-squared: {r2_train_rf:.4f}")

# Calculate and print metrics for Validation Set
rmse_val_rf = np.sqrt(mean_squared_error(y_val, y_val_pred_rf))
r2_val_rf = r2_score(y_val, y_val_pred_rf)

print("\nRandom Forest Performance on Validation Set:")
print(f"  RMSE: {rmse_val_rf:.4f}")
print(f"  R-squared: {r2_val_rf:.4f}")


# Calculate and print metrics for Test Set
rmse_test_rf = np.sqrt(mean_squared_error(y_test, y_test_pred_rf))
r2_test_rf = r2_score(y_test, y_test_pred_rf)

print("\nRandom Forest Performance on Test Set:")
print(f"  RMSE: {rmse_test_rf:.4f}")
print(f"  R-squared: {r2_test_rf:.4f}")

Not bad performance, but it looks like we have some overfitting to training.  Let's do some hyperparameter tuning next to address that.  

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, make_scorer
import numpy as np

# Define a parameter grid for the Random Forest Regressor
# We'll tune n_estimators, max_depth, and min_samples_split for now
param_grid_rf = {
    'n_estimators': [50, 100, 200],  # Number of trees in the forest
    'max_depth': [None, 10, 20],       # Maximum depth of the trees
    'min_samples_split': [15, 20, 25]    # Minimum number of samples required to split an internal node
}

# Create a scorer for negative mean squared error (GridSearchCV maximizes scores)
neg_mse_scorer = make_scorer(mean_squared_error, greater_is_better=False)

# Instantiate GridSearchCV with the Random Forest Regressor and the parameter grid
grid_search_rf = GridSearchCV(RandomForestRegressor(random_state=42), param_grid_rf, cv=5, scoring=neg_mse_scorer, n_jobs=-1)

# Fit GridSearchCV to the training data
grid_search_rf.fit(X_train, y_train)

# Print the best hyperparameters and best cross-validation score
print("Best hyperparameters found for Random Forest:", grid_search_rf.best_params_)

# Convert the best score back to positive MSE or RMSE for easier interpretation
best_mse_rf = -grid_search_rf.best_score_
best_rmse_rf = np.sqrt(best_mse_rf)
print(f"Best cross-validation RMSE for Random Forest: {best_rmse_rf:.4f}")

And now we will re-run the model with the selected parameters.

In [ ]:
# Retrieve the best hyperparameters for the Random Forest
best_params_rf = grid_search_rf.best_params_
print("Best hyperparameters for Random Forest:", best_params_rf)

# Instantiate a new Random Forest Regressor with the best hyperparameters
tuned_rf_model = RandomForestRegressor(**best_params_rf, random_state=42)

# Train the tuned model on the complete training dataset
tuned_rf_model.fit(X_train, y_train)

# Make predictions on training set
y_train_pred_tuned_rf = tuned_rf_model.predict(X_train)

# Make predictions on validation set
y_val_pred_tuned_rf = tuned_rf_model.predict(X_val)

# Make predictions on test set
y_test_pred_tuned_rf = tuned_rf_model.predict(X_test)

# Calculate and print metrics for Training Set
rmse_train_tuned_rf = np.sqrt(mean_squared_error(y_train, y_train_pred_tuned_rf))
r2_train_tuned_rf = r2_score(y_train, y_train_pred_tuned_rf)

print("\nTuned Random Forest Performance on Training Set:")
print(f"  RMSE: {rmse_train_tuned_rf:.4f}")
print(f"  R-squared: {r2_train_tuned_rf:.4f}")

# Calculate and print metrics for Validation Set
rmse_val_tuned_rf = np.sqrt(mean_squared_error(y_val, y_val_pred_tuned_rf))
r2_val_tuned_rf = r2_score(y_val, y_val_pred_tuned_rf)

print("\nTuned Random Forest Performance on Validation Set:")
print(f"  RMSE: {rmse_val_tuned_rf:.4f}")
print(f"  R-squared: {r2_val_tuned_rf:.4f}")

# Calculate and print metrics for Test Set
rmse_test_tuned_rf = np.sqrt(mean_squared_error(y_test, y_test_pred_tuned_rf))
r2_test_tuned_rf = r2_score(y_test, y_test_pred_tuned_rf)

print("\nTuned Random Forest Performance on Test Set:")
print(f"  RMSE: {rmse_test_tuned_rf:.4f}")
print(f"  R-squared: {r2_test_tuned_rf:.4f}")

This is looking better.  Still a bit overfit, but not as bad.  You could try other values when defining the grid parameters to see you can further improve the overfitting.  

Now let's look at the feature performance.

In [ ]:
import pandas as pd

# Get the feature importances from the tuned random forest model
feature_importances_rf = tuned_rf_model.feature_importances_

# Get the feature names from the training data
feature_names = X_train.columns

# Create a pandas Series to easily view feature importances with their names
feature_importance_series_rf = pd.Series(feature_importances_rf, index=feature_names)

# Sort the feature importances in descending order
sorted_feature_importances_rf = feature_importance_series_rf.sort_values(ascending=False)

# Print the sorted feature importances
print("Feature Importances for Tuned Random Forest:")
print(sorted_feature_importances_rf)

Because of the random variable selection, we now have all variables being used in the model at least a little bit.

It's always a good idea to explore our performance visually by looking at the distribution of residuals and the actual by predicted plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate residuals for each set using the tuned Random Forest model
residuals_train = y_train - y_train_pred_tuned_rf
residuals_val = y_val - y_val_pred_tuned_rf
residuals_test = y_test - y_test_pred_tuned_rf

# Create a figure and axes for the plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

# Plot the distribution of residuals for the training set
sns.histplot(residuals_train, kde=True, ax=axes[0])
axes[0].set_title('Training Set Residuals')
axes[0].set_xlabel('Residuals')
axes[0].set_ylabel('Frequency')

# Plot the distribution of residuals for the validation set
sns.histplot(residuals_val, kde=True, ax=axes[1])
axes[1].set_title('Validation Set Residuals')
axes[1].set_xlabel('Residuals')
axes[1].set_ylabel('Frequency')

# Plot the distribution of residuals for the test set
sns.histplot(residuals_test, kde=True, ax=axes[2])
axes[2].set_title('Test Set Residuals')
axes[2].set_xlabel('Residuals')
axes[2].set_ylabel('Frequency')

# Adjust layout to prevent overlapping titles/labels
plt.tight_layout()

# Show the plots
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a figure and axes for the plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot actual vs. predicted for the training set
sns.scatterplot(x=y_train, y=y_train_pred_tuned_rf, ax=axes[0], alpha=0.6)
axes[0].set_title('Training Set: Actual vs. Predicted Price')
axes[0].set_xlabel('Actual Price')
axes[0].set_ylabel('Predicted Price')
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'k--', lw=2) # Add a diagonal line

# Plot actual vs. predicted for the validation set
sns.scatterplot(x=y_val, y=y_val_pred_tuned_rf, ax=axes[1], alpha=0.6)
axes[1].set_title('Validation Set: Actual vs. Predicted Price')
axes[1].set_xlabel('Actual Price')
axes[1].set_ylabel('Predicted Price')
axes[1].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'k--', lw=2) # Add a diagonal line


# Plot actual vs. predicted for the test set
sns.scatterplot(x=y_test, y=y_test_pred_tuned_rf, ax=axes[2], alpha=0.6)
axes[2].set_title('Test Set: Actual vs. Predicted Price')
axes[2].set_xlabel('Actual Price')
axes[2].set_ylabel('Predicted Price')
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2) # Add a diagonal line


# Adjust layout
plt.tight_layout()

# Show the plots
plt.show()

#Boosted Tree

Now let's build a boosted tree model.  maybe with the help of boosting, we can further improve performance. By default it will use a learning rate of 0.1 and build 100 layers (trees).

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

# Instantiate a Gradient Boosting Regressor model
gbr_model = GradientBoostingRegressor(random_state=42)

# Train the model on the training data
gbr_model.fit(X_train, y_train)

# Make predictions on training set
y_train_pred_gbr = gbr_model.predict(X_train)

# Make predictions on validation set
y_val_pred_gbr = gbr_model.predict(X_val)

# Make predictions on test set
y_test_pred_gbr = gbr_model.predict(X_test)

# Calculate and print metrics for Training Set
rmse_train_gbr = np.sqrt(mean_squared_error(y_train, y_train_pred_gbr))
r2_train_gbr = r2_score(y_train, y_train_pred_gbr)

print("Gradient Boosting Regressor Performance on Training Set:")
print(f"  RMSE: {rmse_train_gbr:.4f}")
print(f"  R-squared: {r2_train_gbr:.4f}")

# Calculate and print metrics for Validation Set
rmse_val_gbr = np.sqrt(mean_squared_error(y_val, y_val_pred_gbr))
r2_val_gbr = r2_score(y_val, y_val_pred_gbr)

print("\nGradient Boosting Regressor Performance on Validation Set:")
print(f"  RMSE: {rmse_val_gbr:.4f}")
print(f"  R-squared: {r2_val_gbr:.4f}")


# Calculate and print metrics for Test Set
rmse_test_gbr = np.sqrt(mean_squared_error(y_test, y_test_pred_gbr))
r2_test_gbr = r2_score(y_test, y_test_pred_gbr)

print("\nGradient Boosting Regressor Performance on Test Set:")
print(f"  RMSE: {rmse_test_gbr:.4f}")
print(f"  R-squared: {r2_test_gbr:.4f}")

We have some mild overfitting. Let's try tuning some parameters.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, make_scorer
import numpy as np

# Define a parameter grid for the Gradient Boosting Regressor
# Key parameters to tune include n_estimators, learning_rate, max_depth, and min_samples_split
param_grid_gbr = {
    'n_estimators': [100, 200, 300],  # Number of boosting stages
    'learning_rate': [0.01, 0.1, 0.2], # Shrinkage on the contributions of each tree
    'max_depth': [3, 4, 5],           # Maximum depth of the individual regression estimators
    'min_samples_split': [2, 5, 10]    # Minimum number of samples required to split an internal node
}

# Create a scorer for negative mean squared error (GridSearchCV maximizes scores)
neg_mse_scorer = make_scorer(mean_squared_error, greater_is_better=False)

# Instantiate GridSearchCV with the Gradient Boosting Regressor and the parameter grid
grid_search_gbr = GridSearchCV(GradientBoostingRegressor(random_state=42), param_grid_gbr, cv=5, scoring=neg_mse_scorer, n_jobs=-1)

# Fit GridSearchCV to the training data
grid_search_gbr.fit(X_train, y_train)

# Print the best hyperparameters and best cross-validation score
print("Best hyperparameters found for Gradient Boosting Regressor:", grid_search_gbr.best_params_)

# Convert the best score back to positive MSE or RMSE for easier interpretation
best_mse_gbr = -grid_search_gbr.best_score_
best_rmse_gbr = np.sqrt(best_mse_gbr)
print(f"Best cross-validation RMSE for Gradient Boosting Regressor: {best_rmse_gbr:.4f}")

And we will rerun the model with those parameters.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Retrieve the best hyperparameters for the Gradient Boosting Regressor
best_params_gbr = grid_search_gbr.best_params_
print("Best hyperparameters for Gradient Boosting Regressor:", best_params_gbr)

# Instantiate a new Gradient Boosting Regressor with the best hyperparameters
tuned_gbr_model = GradientBoostingRegressor(**best_params_gbr, random_state=42)

# Train the tuned model on the complete training dataset
tuned_gbr_model.fit(X_train, y_train)

# Make predictions on training set
y_train_pred_tuned_gbr = tuned_gbr_model.predict(X_train)

# Make predictions on validation set
y_val_pred_tuned_gbr = tuned_gbr_model.predict(X_val)

# Make predictions on test set
y_test_pred_tuned_gbr = tuned_gbr_model.predict(X_test)

# Calculate and print metrics for Training Set
rmse_train_tuned_gbr = np.sqrt(mean_squared_error(y_train, y_train_pred_tuned_gbr))
r2_train_tuned_gbr = r2_score(y_train, y_train_pred_tuned_gbr)

print("\nTuned Gradient Boosting Regressor Performance on Training Set:")
print(f"  RMSE: {rmse_train_tuned_gbr:.4f}")
print(f"  R-squared: {r2_train_tuned_gbr:.4f}")

# Calculate and print metrics for Validation Set
rmse_val_tuned_gbr = np.sqrt(mean_squared_error(y_val, y_val_pred_tuned_gbr))
r2_val_tuned_gbr = r2_score(y_val, y_val_pred_tuned_gbr)

print("\nTuned Gradient Boosting Regressor Performance on Validation Set:")
print(f"  RMSE: {rmse_val_tuned_gbr:.4f}")
print(f"  R-squared: {r2_val_tuned_gbr:.4f}")

# Calculate and print metrics for Test Set
rmse_test_tuned_gbr = np.sqrt(mean_squared_error(y_test, y_test_pred_tuned_gbr))
r2_test_tuned_gbr = r2_score(y_test, y_test_pred_tuned_gbr)

print("\nTuned Gradient Boosting Regressor Performance on Test Set:")
print(f"  RMSE: {rmse_test_tuned_gbr:.4f}")
print(f"  R-squared: {r2_test_tuned_gbr:.4f}")

Now let's check our feature importance.

In [ ]:
import pandas as pd

# Get the feature importances from the tuned gradient boosting regressor model
feature_importances_gbr = tuned_gbr_model.feature_importances_

# Get the feature names from the training data
feature_names = X_train.columns

# Create a pandas Series to easily view feature importances with their names
feature_importance_series_gbr = pd.Series(feature_importances_gbr, index=feature_names)

# Sort the feature importances in descending order
sorted_feature_importances_gbr = feature_importance_series_gbr.sort_values(ascending=False)

# Print the sorted feature importances
print("Feature Importances for Tuned Gradient Boosting Regressor:")
print(sorted_feature_importances_gbr)

And lastly our visualizations of model performance.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate residuals for each set using the tuned Gradient Boosting Regressor model
residuals_train_gbr = y_train - y_train_pred_tuned_gbr
residuals_val_gbr = y_val - y_val_pred_tuned_gbr
residuals_test_gbr = y_test - y_test_pred_tuned_gbr

# Create a figure and axes for the plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

# Plot the distribution of residuals for the training set
sns.histplot(residuals_train_gbr, kde=True, ax=axes[0])
axes[0].set_title('Training Set Residuals (Tuned GBR)')
axes[0].set_xlabel('Residuals')
axes[0].set_ylabel('Frequency')

# Plot the distribution of residuals for the validation set
sns.histplot(residuals_val_gbr, kde=True, ax=axes[1])
axes[1].set_title('Validation Set Residuals (Tuned GBR)')
axes[1].set_xlabel('Residuals')
axes[1].set_ylabel('Frequency')

# Plot the distribution of residuals for the test set
sns.histplot(residuals_test_gbr, kde=True, ax=axes[2])
axes[2].set_title('Test Set Residuals (Tuned GBR)')
axes[2].set_xlabel('Residuals')
axes[2].set_ylabel('Frequency')

# Adjust layout to prevent overlapping titles/labels
plt.tight_layout()

# Show the plots
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a figure and axes for the plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot actual vs. predicted for the training set
sns.scatterplot(x=y_train, y=y_train_pred_tuned_gbr, ax=axes[0], alpha=0.6)
axes[0].set_title('Training Set: Actual vs. Predicted Price (Tuned GBR)')
axes[0].set_xlabel('Actual Price')
axes[0].set_ylabel('Predicted Price')
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'k--', lw=2) # Add a diagonal line

# Plot actual vs. predicted for the validation set
sns.scatterplot(x=y_val, y=y_val_pred_tuned_gbr, ax=axes[1], alpha=0.6)
axes[1].set_title('Validation Set: Actual vs. Predicted Price (Tuned GBR)')
axes[1].set_xlabel('Actual Price')
axes[1].set_ylabel('Predicted Price')
axes[1].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'k--', lw=2) # Add a diagonal line


# Plot actual vs. predicted for the test set
sns.scatterplot(x=y_test, y=y_test_pred_tuned_gbr, ax=axes[2], alpha=0.6)
axes[2].set_title('Test Set: Actual vs. Predicted Price (Tuned GBR)')
axes[2].set_xlabel('Actual Price')
axes[2].set_ylabel('Predicted Price')
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2) # Add a diagonal line


# Adjust layout
plt.tight_layout()

# Show the plots
plt.show()

#Model Comparison

We built three different models.  Let's see how they compare.  

In [ ]:
import pandas as pd

# Collect performance metrics for each tuned model

# Tuned Regression Tree
metrics_reg = {
    'Model': 'Tuned Regression Tree',
    'RMSE (Train)': rmse_train_tuned_reg,
    'R2 (Train)': r2_train_tuned_reg,
    'RMSE (Validation)': rmse_val_tuned_reg,
    'R2 (Validation)': r2_val_tuned_reg,
    'RMSE (Test)': rmse_test_tuned_reg,
    'R2 (Test)': r2_test_tuned_reg
}

# Tuned Random Forest
metrics_rf = {
    'Model': 'Tuned Random Forest',
    'RMSE (Train)': rmse_train_tuned_rf,
    'R2 (Train)': r2_train_tuned_rf,
    'RMSE (Validation)': rmse_val_tuned_rf,
    'R2 (Validation)': r2_val_tuned_rf,
    'RMSE (Test)': rmse_test_tuned_rf,
    'R2 (Test)': r2_test_tuned_rf
}

# Tuned Gradient Boosting Regressor
metrics_gbr = {
    'Model': 'Tuned Gradient Boosting Regressor',
    'RMSE (Train)': rmse_train_tuned_gbr,
    'R2 (Train)': r2_train_tuned_gbr,
    'RMSE (Validation)': rmse_val_tuned_gbr,
    'R2 (Validation)': r2_val_tuned_gbr,
    'RMSE (Test)': rmse_test_tuned_gbr,
    'R2 (Test)': r2_test_tuned_gbr
}

# Create a list of metric dictionaries
all_metrics = [metrics_reg, metrics_rf, metrics_gbr]

# Create a pandas DataFrame
metrics_df = pd.DataFrame(all_metrics)

# Set 'Model' as the index for better readability
metrics_df = metrics_df.set_index('Model')

# Round the numeric columns to 2 decimal places
metrics_df_rounded = metrics_df.round(2)


# Display the comparison table
print("Model Performance Comparison:")
display(metrics_df_rounded)

These are all pretty close!  But it's clear that both of the ensemble tree models outperformed the basic regression tree.  Overall, I'd probably select the Boosted Tree model as the best.  There is also room to try different parameter values to eek out even better performance.  Modeling work is never truly done...  There's always something else to consider trying!